# OPTN Kidney Waitlist Data Cleaning Pipeline

This notebook creates a reproducible, model-independent cleaned version of the OPTN kidney waitlist analytic dataset.

## Go to bottom for reusbale cleaning function

The pipeline:
- preserves the raw dataframe
- converts date fields to datetime
- flags questionable BMI values
- flags invalid date records
- handles an invalid functional status code
- assigns appropriate categorical and ID data types
- documents structured missingness
- creates missingness indicators for important variables
- validates key data-quality assumptions

Model-specific steps such as train/test splitting, encoding, scaling, and training-set-based imputation are intentionally kept separate.

In [1]:
!git clone https://github.com/Break-Through-Tech/Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation.git

Cloning into 'Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 149 (delta 11), reused 6 (delta 3), pack-reused 128 (from 2)
Receiving objects: 100% (149/149), 23.18 MiB | 37.03 MiB/s, done.
Resolving deltas: 100% (72/72), done.


In [2]:
%cd Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation


/content/Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation


In [3]:
!git branch -a

* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/Julia-Zhao
  remotes/origin/Prince-Subedi
  remotes/origin/arjunmagar
  remotes/origin/main


In [4]:
!git checkout Julia-Zhao

branch 'Julia-Zhao' set up to track 'origin/Julia-Zhao'.
Switched to a new branch 'Julia-Zhao'


In [5]:
!git branch

* Julia-Zhao
  main


In [6]:
!git config user.name "Julia Zhao"
!git config user.email "juliazhaott@gmail.com"

In [7]:
import pandas as pd
import numpy as np

In [19]:
file_path = "data/kidney_waitlist_analytic.csv.gz"

df = pd.read_csv(
    file_path,
    compression="gzip",
    low_memory=False
)

print(df_clean.shape)
df.head()

(494862, 38)


,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,END_CPRA,REM_CD,...,PSTATUS,PTIME,TRR_ID_CODE,DONOR_ID,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,13609,died,1,0,0,287.0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,NaN,24.0,...,NaN,NaN,NaN,NaN,6975,removed_administrative,0,0,0,2322.0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,19716,died,1,0,0,604.0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,8587,removed_too_sick,1,0,0,909.0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,18352,removed_too_sick,1,0,0,231.0


### Preserve the raw dataframe

In [20]:
# Never modify the original dataframe directly.
df_clean = df.copy()

print("Working copy created:", df_clean.shape)

Working copy created: (494862, 38)


### Convert date columns

In [21]:
# Convert date fields from strings/objects into proper datetime values.

date_cols = [
    "INIT_DATE",
    "END_DATE",
    "DIALYSIS_DATE",
    "TX_DATE",
    "COMPOSITE_DEATH_DATE"
]

for col in date_cols:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

print(df_clean[date_cols].dtypes)

INIT_DATE               datetime64[ns]
END_DATE                datetime64[ns]
DIALYSIS_DATE           datetime64[ns]
TX_DATE                 datetime64[ns]
COMPOSITE_DEATH_DATE    datetime64[ns]
dtype: object


### Create data-quality flags

In [23]:
# BMI:
# The EDA used <=10 and >60 as a descriptive questionable range.
# We FLAG these values rather than automatically deleting them because
# the final clinically defensible range should be confirmed with the advisor.

df_clean["BMI_INVALID"] = (
    (df_clean["BMI_TCR"] <= 10) |
    (df_clean["BMI_TCR"] > 60)
)


# Invalid date:
# END_DATE occurring before INIT_DATE is logically inconsistent.

df_clean["INVALID_DATE"] = (
    df_clean["END_DATE"] < df_clean["INIT_DATE"]
)


# Invalid functional status:
# One record contains FUNC_STAT_TCR = 1, which does not match
# the expected coding scheme identified in the EDA.

df_clean["FUNC_STAT_INVALID"] = (
    df_clean["FUNC_STAT_TCR"] == 1
)


print("Questionable BMI values:", df_clean["BMI_INVALID"].sum())
print("Invalid dates:", df_clean["INVALID_DATE"].sum())
print("Invalid functional status:", df_clean["FUNC_STAT_INVALID"].sum())

Questionable BMI values: 714
Invalid dates: 59
Invalid functional status: 1


### Clean invalid functional-status value

In [24]:
# Preserve the row but replace the unsupported functional-status code with missing.

df_clean.loc[
    df_clean["FUNC_STAT_INVALID"],
    "FUNC_STAT_TCR"
] = np.nan

print(
    "Remaining FUNC_STAT_TCR == 1:",
    (df_clean["FUNC_STAT_TCR"] == 1).sum()
)

Remaining FUNC_STAT_TCR == 1: 0


### Add missingness indicators

In [26]:
# Preserve original missing values while making important missingness explicit.
# We intentionally do not impute INIT_CPRA here because the EDA found essentially
#0% missing through 2022 and roughly 80% missing beginning in 2023. That is a structural temporal shift, not ordinary random missingness.

df_clean["BMI_MISSING"] = df_clean["BMI_TCR"].isna()

df_clean["FUNC_STAT_MISSING"] = (
    df_clean["FUNC_STAT_TCR"].isna()
)

df_clean["INIT_CPRA_MISSING"] = (
    df_clean["INIT_CPRA"].isna()
)

## Document structured DIALYSIS_DATE missingness

In [27]:
# Check whether dialysis-date missingness matches dialysis status.

dialysis_missingness = pd.crosstab(
    df_clean["ON_DIALYSIS"],
    df_clean["DIALYSIS_DATE"].isna(),
    margins=True
)

dialysis_missingness

DIALYSIS_DATE,False,True,All
ON_DIALYSIS,,,
N,0,134245,134245
Y,360617,0,360617
All,360617,134245,494862


In [28]:
# Explicit flag showing whether a dialysis date is available.
# We DO NOT impute DIALYSIS_DATE.

df_clean["DIALYSIS_DATE_AVAILABLE"] = (
    df_clean["DIALYSIS_DATE"].notna()
)

### Document A2A2B_ELIGIBILITY applicability

In [29]:
# A2/A2B eligibility is primarily applicable to Blood Type B candidates.
# Missing values for non-B candidates should not automatically be interpreted as "No".

df_clean["A2A2B_APPLICABLE"] = (
    df_clean["ABO"] == "B"
)

In [30]:
pd.crosstab(
    df_clean["ABO"],
    df_clean["A2A2B_ELIGIBILITY"].isna()
)

A2A2B_ELIGIBILITY,False,True
ABO,,
A,0,155594
A1,0,2398
A1B,0,226
A2,0,365
A2B,0,75
AB,0,18475
B,38258,35778
O,0,243693


Convert identifiers to string

In [31]:
# IDs are labels, not continuous numeric measurements.

id_cols = [
    "PT_CODE",
    "WL_ID_CODE",
    "LISTING_CTR_CODE"
]

for col in id_cols:
    df_clean[col] = df_clean[col].astype("string")

print(df_clean[id_cols].dtypes)

PT_CODE             string[python]
WL_ID_CODE          string[python]
LISTING_CTR_CODE    string[python]
dtype: object


Convert coded variables to categorical

In [32]:
# These variables represent groups/codes rather than continuous quantities.

categorical_cols = [
    "GENDER",
    "ETHCAT",
    "ABO",
    "ON_DIALYSIS",
    "FUNC_STAT_TCR",
    "INIT_STAT",
    "REGION"
]

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype("category")

print(df_clean[categorical_cols].dtypes)

GENDER           category
ETHCAT           category
ABO              category
ON_DIALYSIS      category
FUNC_STAT_TCR    category
INIT_STAT        category
REGION           category
dtype: object


## Basic range and consistency validation

In [33]:
print("=== DATA QUALITY VALIDATION ===")

print(
    "Duplicate WL_ID_CODE:",
    df_clean["WL_ID_CODE"].duplicated().sum()
)

print(
    "Age < 0:",
    (df_clean["INIT_AGE"] < 0).sum()
)

print(
    "Age > 100:",
    (df_clean["INIT_AGE"] > 100).sum()
)

cpra_out_of_range = (
    (df_clean["INIT_CPRA"] < 0) |
    (df_clean["INIT_CPRA"] > 100)
)

print(
    "INIT_CPRA outside 0-100:",
    cpra_out_of_range.sum()
)

print(
    "Invalid dates:",
    df_clean["INVALID_DATE"].sum()
)

print(
    "Questionable BMI:",
    df_clean["BMI_INVALID"].sum()
)

=== DATA QUALITY VALIDATION ===
Duplicate WL_ID_CODE: 0
Age < 0: 0
Age > 100: 0
INIT_CPRA outside 0-100: 0
Invalid dates: 59
Questionable BMI: 714


# Final cleaning summary

In [34]:
cleaning_summary = pd.DataFrame({
    "Issue": [
        "Questionable BMI (<=10 or >60)",
        "END_DATE before INIT_DATE",
        "Invalid FUNC_STAT_TCR = 1",
        "Missing BMI_TCR",
        "Missing FUNC_STAT_TCR",
        "Missing INIT_CPRA"
    ],

    "Count": [
        df_clean["BMI_INVALID"].sum(),
        df_clean["INVALID_DATE"].sum(),
        df_clean["FUNC_STAT_INVALID"].sum(),
        df_clean["BMI_TCR"].isna().sum(),
        df_clean["FUNC_STAT_TCR"].isna().sum(),
        df_clean["INIT_CPRA"].isna().sum()
    ],

    "Action": [
        "Flagged; original value preserved pending domain guidance",
        "Flagged; row preserved",
        "Converted value to missing",
        "Preserved; imputation deferred until after train/test split",
        "Preserved; imputation deferred until after train/test split",
        "Preserved; structural temporal missingness documented"
    ]
})

cleaning_summary

,Issue,Count,Action
0,Questionable BMI (<=10 or >60),714,Flagged; original value preserved pending doma...
1,END_DATE before INIT_DATE,59,Flagged; row preserved
2,Invalid FUNC_STAT_TCR = 1,1,Converted value to missing
3,Missing BMI_TCR,1558,Preserved; imputation deferred until after tra...
4,Missing FUNC_STAT_TCR,3976,Preserved; imputation deferred until after tra...
5,Missing INIT_CPRA,142741,Preserved; structural temporal missingness doc...


# **Final reusable cleaning function**

In [35]:
def clean_waitlist_data(df):
    """
    Apply model-independent cleaning to the OPTN kidney waitlist dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw OPTN kidney waitlist analytic dataframe.

    Returns
    -------
    pandas.DataFrame
        Cleaned dataframe with:
        - converted date columns
        - data-quality flags
        - missingness indicators
        - structured-missingness indicators
        - corrected unsupported FUNC_STAT_TCR value
        - identifier and categorical dtypes

    Notes
    -----
    This function intentionally DOES NOT:
    - impute INIT_CPRA
    - remove censored patients
    - choose the classification cohort
    - select model features
    - one-hot encode variables
    - standardize variables
    - create train/test splits
    - calculate imputation statistics from the full dataset
    """

    df_clean = df.copy()

    # -----------------------------
    # Date conversion
    # -----------------------------

    date_cols = [
        "INIT_DATE",
        "END_DATE",
        "DIALYSIS_DATE",
        "TX_DATE",
        "COMPOSITE_DEATH_DATE"
    ]

    for col in date_cols:
        df_clean[col] = pd.to_datetime(
            df_clean[col],
            errors="coerce"
        )

    # -----------------------------
    # Data-quality flags
    # -----------------------------

    df_clean["BMI_INVALID"] = (
        (df_clean["BMI_TCR"] <= 10) |
        (df_clean["BMI_TCR"] > 60)
    )

    df_clean["INVALID_DATE"] = (
        df_clean["END_DATE"] < df_clean["INIT_DATE"]
    )

    df_clean["FUNC_STAT_INVALID"] = (
        df_clean["FUNC_STAT_TCR"] == 1
    )

    # Replace unsupported functional-status value with missing.
    df_clean.loc[
        df_clean["FUNC_STAT_INVALID"],
        "FUNC_STAT_TCR"
    ] = np.nan

    # -----------------------------
    # Missingness indicators
    # -----------------------------

    df_clean["BMI_MISSING"] = (
        df_clean["BMI_TCR"].isna()
    )

    df_clean["FUNC_STAT_MISSING"] = (
        df_clean["FUNC_STAT_TCR"].isna()
    )

    df_clean["INIT_CPRA_MISSING"] = (
        df_clean["INIT_CPRA"].isna()
    )

    # -----------------------------
    # Structured missingness
    # -----------------------------

    df_clean["DIALYSIS_DATE_AVAILABLE"] = (
        df_clean["DIALYSIS_DATE"].notna()
    )

    df_clean["A2A2B_APPLICABLE"] = (
        df_clean["ABO"] == "B"
    )

    # -----------------------------
    # Identifier types
    # -----------------------------

    id_cols = [
        "PT_CODE",
        "WL_ID_CODE",
        "LISTING_CTR_CODE"
    ]

    for col in id_cols:
        df_clean[col] = df_clean[col].astype("string")

    # -----------------------------
    # Categorical types
    # -----------------------------

    categorical_cols = [
        "GENDER",
        "ETHCAT",
        "ABO",
        "ON_DIALYSIS",
        "FUNC_STAT_TCR",
        "INIT_STAT",
        "REGION"
    ]

    for col in categorical_cols:
        df_clean[col] = df_clean[col].astype("category")

    return df_clean

In [36]:
df_final = clean_waitlist_data(df)

print("Raw shape:", df.shape)
print("Clean shape:", df_final.shape)

print("\nQuality flags:")
print(
    df_final[
        [
            "BMI_INVALID",
            "INVALID_DATE",
            "FUNC_STAT_INVALID"
        ]
    ].sum()
)

print("\nImportant missingness:")
print(
    df_final[
        [
            "BMI_TCR",
            "FUNC_STAT_TCR",
            "INIT_CPRA"
        ]
    ].isna().sum()
)


Raw shape: (494862, 38)
Clean shape: (494862, 46)

Quality flags:
BMI_INVALID          714
INVALID_DATE          59
FUNC_STAT_INVALID      1
dtype: int64

Important missingness:
BMI_TCR            1558
FUNC_STAT_TCR      3976
INIT_CPRA        142741
dtype: int64


## Cleaning Decisions Intentionally Deferred

Some preprocessing decisions are intentionally not hard-coded into the canonical cleaning pipeline:

- **BMI cutoff:** Values <=10 or >60 are currently flagged based on the EDA. The final clinically defensible range should be confirmed with the advisor/domain experts.
- **INIT_CPRA:** Missing values are preserved because missingness changes sharply beginning in 2023. CPRA handling should be evaluated separately.
- **Censored patients:** Still-waiting registrations are preserved in the canonical dataset. Candidate classification cohorts can be generated separately.
- **DIALYSIS_DATE:** Missingness is structurally related to ON_DIALYSIS and is therefore not imputed.
- **A2A2B_ELIGIBILITY:** Missingness is partly due to the field only being applicable to certain blood-type B candidates.
- **Imputation:** Numerical imputation values must be learned from training data only after the patient-grouped train/validation/test split.
- **Encoding and scaling:** These are model-specific preprocessing steps and are not part of this cleaning function.
- **Feature selection:** Leakage-safe feature selection is performed separately from data cleaning.